# MODULE 10 — Interview Problems
## 25 Real Pandas Interview Questions — With Solutions

---
*These are taken from actual interviews at FAANG, startups, and BI-heavy companies.*

> **Format:** Problem statement → Hint → Solution → Explanation

In [ ]:
import pandas as pd
import numpy as np
warnings = __import__('warnings'); warnings.filterwarnings('ignore')

# ── Setup: shared data for all interview problems ─────────────────────────────
np.random.seed(42)
n = 100

orders = pd.DataFrame({
    'order_id'   : range(1, n+1),
    'customer_id': np.random.randint(1, 21, n),
    'product'    : np.random.choice(['A','B','C','D'], n),
    'revenue'    : np.random.uniform(50, 2000, n).round(2),
    'quantity'   : np.random.randint(1, 6, n),
    'date'       : pd.date_range('2024-01-01', periods=n, freq='3D'),
    'region'     : np.random.choice(['North','South','East','West'], n)
})

employees = pd.DataFrame({
    'emp_id'    : range(1, 11),
    'name'      : ['Alice','Bob','Carol','David','Eve',
                   'Frank','Grace','Henry','Ivy','Jack'],
    'dept'      : ['Sales','Sales','Eng','Eng','HR',
                   'Sales','Eng','HR','Sales','Eng'],
    'salary'    : [70000,80000,95000,110000,60000,
                   75000,105000,65000,72000,98000],
    'manager_id': [None,1,None,3,None,1,3,5,1,3]
})

print('Datasets ready. Starting interview problems...')

---
### Q1: Find the top 3 products by total revenue

In [ ]:
# Solution
top3 = (orders
    .groupby('product')['revenue']
    .sum()
    .nlargest(3)
    .reset_index()
    .rename(columns={'revenue': 'total_revenue'})
)
print(top3)
# Key: nlargest() is cleaner than sort_values().head()

---
### Q2: Find customers who placed more than 5 orders

In [ ]:
frequent = (
    orders
    .groupby('customer_id')['order_id']
    .count()
    .reset_index()
    .rename(columns={'order_id': 'order_count'})
    .query('order_count > 5')
    .sort_values('order_count', ascending=False)
)
print(frequent)

---
### Q3: Calculate the running total (cumulative sum) of revenue by date

In [ ]:
daily_rev = orders.groupby('date')['revenue'].sum().reset_index()
daily_rev['cumulative_revenue'] = daily_rev['revenue'].cumsum().round(2)
print(daily_rev.head(10).to_string(index=False))

---
### Q4: Find the month-over-month revenue growth rate

In [ ]:
orders['month'] = orders['date'].dt.to_period('M')
monthly = orders.groupby('month')['revenue'].sum()
mom_growth = monthly.pct_change() * 100
result = pd.DataFrame({'revenue': monthly, 'mom_pct': mom_growth.round(1)})
print(result.to_string())

---
### Q5: Find employees who earn above their department average

In [ ]:
# transform() keeps same index as original df — key insight
employees['dept_avg_salary'] = employees.groupby('dept')['salary'].transform('mean')
above_avg = employees[employees['salary'] > employees['dept_avg_salary']]
print(above_avg[['name', 'dept', 'salary', 'dept_avg_salary']].to_string(index=False))

---
### Q6: Pivot — show revenue by region and product (matrix form)

In [ ]:
pivot = orders.pivot_table(
    values='revenue', index='region', columns='product',
    aggfunc='sum', fill_value=0
).round(0)
print(pivot.to_string())

---
### Q7: Find the first and last order date per customer

In [ ]:
customer_dates = orders.groupby('customer_id')['date'].agg(
    first_order='min',
    last_order ='max'
).reset_index()
customer_dates['active_days'] = (customer_dates['last_order'] - customer_dates['first_order']).dt.days
print(customer_dates.head(8).to_string(index=False))

---
### Q8: Rank customers by revenue within each region

In [ ]:
cust_region = orders.groupby(['region', 'customer_id'])['revenue'].sum().reset_index()

# rank within group
cust_region['rank'] = cust_region.groupby('region')['revenue'].rank(
    ascending=False, method='dense'
).astype(int)

# Top customer per region
top_per_region = cust_region[cust_region['rank'] == 1].sort_values('region')
print('Top customer per region:')
print(top_per_region.to_string(index=False))

---
### Q9: Detect and remove outliers using IQR

In [ ]:
Q1 = orders['revenue'].quantile(0.25)
Q3 = orders['revenue'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

no_outliers = orders[orders['revenue'].between(lower_bound, upper_bound)]
print(f'Original: {len(orders)} rows | After IQR filter: {len(no_outliers)} rows')
print(f'Removed: {len(orders) - len(no_outliers)} outliers')

---
### Q10: Self-join — find employee-manager pairs

In [ ]:
manager_pairs = employees.merge(
    employees[['emp_id', 'name']].rename(columns={'emp_id':'manager_id', 'name':'manager_name'}),
    on='manager_id',
    how='left'
)
print(manager_pairs[['name', 'dept', 'salary', 'manager_name']].to_string(index=False))

---
### Q11: Reshape wide to long format (melt)

In [ ]:
# Wide format (as it often comes from Excel)
wide = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Carol'],
    'Jan_rev' : [1000, 1500, 900],
    'Feb_rev' : [1200, 1100, 1400],
    'Mar_rev' : [1300, 1600, 800]
})

# Melt to long format (what databases and ML models prefer)
long = wide.melt(id_vars='customer',
                 var_name='month',
                 value_name='revenue')
long['month'] = long['month'].str.replace('_rev', '')
print(long.sort_values('customer').to_string(index=False))

---
### Q12: Reshape long to wide format (pivot)

In [ ]:
# Convert the long format back to wide
back_to_wide = long.pivot(index='customer', columns='month', values='revenue')
print(back_to_wide.to_string())
# pivot() vs pivot_table(): use pivot_table when there are duplicate index/column pairs

---
### Q13: Calculate 7-day rolling average revenue

In [ ]:
daily = orders.groupby('date')['revenue'].sum().reset_index()
daily['7d_ma'] = daily['revenue'].rolling(window=7, min_periods=1).mean().round(2)
print(daily.tail(10).to_string(index=False))

---
### Q14: Find duplicate rows and show which columns differ

In [ ]:
# Create dataset with duplicates
df_with_dupes = pd.DataFrame({
    'id'     : [1, 2, 3, 2, 4, 3],
    'name'   : ['Alice','Bob','Carol','Bob','David','Carol'],
    'revenue': [100, 200, 300, 200, 400, 305]  # note: 305 ≠ 300 → near-duplicate
})

# Exact duplicates on key columns
exact_dupes = df_with_dupes[df_with_dupes.duplicated(subset=['id', 'name'], keep=False)]
print('Rows that share id+name:')
print(exact_dupes.to_string(index=False))

---
### Q15: Calculate percentage of total within each group

In [ ]:
# Revenue % contribution of each product within each region
orders['region_total'] = orders.groupby('region')['revenue'].transform('sum')
orders['pct_of_region'] = (orders['revenue'] / orders['region_total'] * 100).round(2)

# Aggregated view
region_product_pct = (orders
    .groupby(['region', 'product'])['revenue']
    .sum()
    .groupby(level=0)
    .transform(lambda x: x / x.sum() * 100)
    .round(1)
    .reset_index()
    .rename(columns={'revenue': 'pct_of_region_rev'})
)
print(region_product_pct.to_string(index=False))

---
### Q16: Anti-join — find records in A not in B

In [ ]:
all_customers = pd.DataFrame({'customer_id': range(1, 21)})
active_customers = orders['customer_id'].unique()

# Customers who never ordered
inactive = all_customers[~all_customers['customer_id'].isin(active_customers)]
print('Customers with no orders:', inactive['customer_id'].tolist())

# Alternative: using merge with indicator
merged = all_customers.merge(
    orders[['customer_id']].drop_duplicates(),
    on='customer_id', how='left', indicator=True
)
inactive2 = merged[merged['_merge'] == 'left_only']['customer_id']
print('Same result:', inactive2.tolist())

---
### Q17: String operations — extract domain from email

In [ ]:
emails = pd.Series(['alice@gmail.com', 'bob@company.org',
                    'carol@yahoo.com', 'david@startup.io'])

emails_df = pd.DataFrame({'email': emails})
emails_df['username'] = emails_df['email'].str.split('@').str[0]
emails_df['domain']   = emails_df['email'].str.split('@').str[1]
emails_df['tld']      = emails_df['email'].str.split('.').str[-1]
emails_df['is_free_email'] = emails_df['domain'].isin(['gmail.com','yahoo.com','hotmail.com'])

print(emails_df.to_string(index=False))

---
### Q18: Handle time zones in datetime data

In [ ]:
# Timestamps from different offices
ts_utc = pd.Timestamp('2024-06-15 14:30:00', tz='UTC')
ts_ny  = ts_utc.tz_convert('America/New_York')
ts_lon = ts_utc.tz_convert('Europe/London')
ts_tok = ts_utc.tz_convert('Asia/Tokyo')

print(f'UTC     : {ts_utc}')
print(f'New York: {ts_ny}')
print(f'London  : {ts_lon}')
print(f'Tokyo   : {ts_tok}')

# Remove timezone info (for DB storage)
print(f'Naive   : {ts_ny.tz_localize(None)}')

---
### Q19: Fill missing values using group-wise statistics

In [ ]:
df_missing = pd.DataFrame({
    'product': ['A','A','B','B','A','B','A'],
    'price'  : [100, np.nan, 200, np.nan, 120, 210, np.nan]
})

print('Before:')
print(df_missing)

# Fill with product-level median (NOT global median)
df_missing['price'] = df_missing.groupby('product')['price'].transform(
    lambda x: x.fillna(x.median())
)
print('\nAfter (group-wise fill):')
print(df_missing)
# Group A median = 110, Group B median = 205

---
### Q20: Window functions — running rank and percentile within group

In [ ]:
sales_df = orders.groupby(['region', 'customer_id'])['revenue'].sum().reset_index()

# Rank within region
sales_df['rank_in_region'] = sales_df.groupby('region')['revenue'].rank(
    ascending=False, method='min'
).astype(int)

# Percentile within region
sales_df['percentile_in_region'] = sales_df.groupby('region')['revenue'].rank(pct=True).round(2)

print(sales_df.sort_values(['region','rank_in_region']).head(12).to_string(index=False))

---
### Q21: Conditional aggregation (SQL CASE WHEN equivalent)

In [ ]:
# Count orders by category AND revenue tier in one pass
result = orders.groupby('product').agg(
    total_orders   = ('order_id', 'count'),
    high_rev_orders= ('revenue', lambda x: (x >= 1000).sum()),   # SQL: COUNT(CASE WHEN rev>=1000)
    low_rev_orders = ('revenue', lambda x: (x < 500).sum()),
    total_revenue  = ('revenue', 'sum')
).round(2)
result['high_rev_pct'] = (result['high_rev_orders'] / result['total_orders'] * 100).round(1)
print(result.to_string())

---
### Q22: Lag and Lead — time series differences

In [ ]:
daily_orders = orders.groupby('date')['revenue'].sum().reset_index().sort_values('date')

# Lag: previous period value
daily_orders['prev_day_rev'] = daily_orders['revenue'].shift(1)   # LAG(1)
daily_orders['prev_week_rev']= daily_orders['revenue'].shift(7)   # LAG(7)

# Lead: next period value
daily_orders['next_day_rev'] = daily_orders['revenue'].shift(-1)  # LEAD(1)

# Day-over-day change
daily_orders['dod_change'] = daily_orders['revenue'] - daily_orders['prev_day_rev']
daily_orders['dod_pct']    = (daily_orders['dod_change'] / daily_orders['prev_day_rev'] * 100).round(1)

print(daily_orders[['date','revenue','prev_day_rev','dod_pct']].head(12).to_string(index=False))

---
### Q23: Explode — one row per list element

In [ ]:
# Common with API data or multi-valued fields
df_tags = pd.DataFrame({
    'order_id': [1, 2, 3],
    'tags'    : [['urgent', 'gift'], ['bulk'], ['corporate', 'net30', 'reseller']]
})
print('Before explode:')
print(df_tags)

exploded = df_tags.explode('tags').reset_index(drop=True)
print('\nAfter explode:')
print(exploded.to_string(index=False))

---
### Q24: Memory-efficient groupby on a large DataFrame

In [ ]:
# Categorical dtypes make groupby dramatically faster
big = pd.DataFrame({
    'region' : pd.Categorical(np.random.choice(['N','S','E','W'], 1_000_000)),
    'product': pd.Categorical(np.random.choice(['A','B','C','D'], 1_000_000)),
    'revenue': np.random.uniform(10, 1000, 1_000_000)
})

import time
t = time.time()
result = big.groupby(['region','product'], observed=True)['revenue'].sum()
print(f'Groupby on 1M rows with Categorical: {time.time()-t:.3f}s')
print(result)

---
### Q25: Build a summary report as a formatted string (for email/Slack)

In [ ]:
def generate_weekly_report(df, report_date='2024-03-31'):
    """Generate a formatted weekly KPI report string."""
    report_date = pd.Timestamp(report_date)
    week_start  = report_date - pd.Timedelta(days=6)
    
    week_df = df[(df['date'] >= week_start) & (df['date'] <= report_date)]
    
    total_rev   = week_df['revenue'].sum()
    total_orders= len(week_df)
    aov         = week_df['revenue'].mean()
    best_product= week_df.groupby('product')['revenue'].sum().idxmax()
    best_region = week_df.groupby('region')['revenue'].sum().idxmax()
    
    report = f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  WEEKLY SALES REPORT — {report_date.strftime('%b %d, %Y')}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Total Revenue  : ${total_rev:>10,.0f}
  Total Orders   : {total_orders:>10,}
  Avg Order Value: ${aov:>10,.0f}
  Best Product   : {best_product}
  Best Region    : {best_region}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
    return report

print(generate_weekly_report(orders))

# MODULE 11 — Case Studies & Practical Exercises
---

## CASE STUDY 1: E-Commerce Funnel Analysis

**Business Context:**  
Your e-commerce team wants to know where customers drop off in the purchase funnel:  
`Visit → Product View → Add to Cart → Checkout → Purchase`

In [ ]:
np.random.seed(99)

# Simulate funnel data
n_users = 10_000
funnel = pd.DataFrame({'user_id': range(1, n_users+1)})

# Each stage, some users drop off
funnel['visited']       = True
funnel['viewed_product']= np.random.random(n_users) < 0.65   # 65% of visitors
funnel['added_to_cart'] = funnel['viewed_product'] & (np.random.random(n_users) < 0.45)
funnel['checked_out']   = funnel['added_to_cart']  & (np.random.random(n_users) < 0.60)
funnel['purchased']     = funnel['checked_out']    & (np.random.random(n_users) < 0.75)

# Funnel analysis
stages = ['visited', 'viewed_product', 'added_to_cart', 'checked_out', 'purchased']
counts = funnel[stages].sum()

funnel_df = pd.DataFrame({
    'stage'       : stages,
    'users'       : counts.values,
})
funnel_df['pct_of_total']  = (funnel_df['users'] / funnel_df['users'].iloc[0] * 100).round(1)
funnel_df['pct_of_prev']   = (funnel_df['users'] / funnel_df['users'].shift(1) * 100).round(1)
funnel_df['drop_off_users']= funnel_df['users'].shift(1) - funnel_df['users']

print('=== E-COMMERCE CONVERSION FUNNEL ===')
print(funnel_df.to_string(index=False))

# Biggest drop-off
worst_stage = funnel_df.loc[funnel_df['pct_of_prev'].idxmin(), 'stage']
print(f'\nBiggest drop-off stage: {worst_stage}')
print('Recommendation: Focus CRO efforts on this stage.')

## CASE STUDY 2: Churn Prediction Feature Engineering

**Business Context:**  
Build a feature table for a churn prediction model.  
Each row = one customer, features = behavioral signals.

In [ ]:
np.random.seed(42)
n_cust = 500
n_orders_total = 3000

transactions = pd.DataFrame({
    'customer_id': np.random.randint(1, n_cust+1, n_orders_total),
    'order_date' : pd.Timestamp('2024-01-01') + 
                   pd.to_timedelta(np.random.randint(0, 365, n_orders_total), unit='D'),
    'revenue'    : np.random.uniform(20, 500, n_orders_total).round(2),
    'category'   : np.random.choice(['Electronics','Clothing','Food','Sports'], n_orders_total)
})

snapshot = pd.Timestamp('2024-12-31')

# Feature engineering for churn model
features = transactions.groupby('customer_id').agg(
    # Recency features
    days_since_last_order = ('order_date', lambda x: (snapshot - x.max()).days),
    days_since_first_order= ('order_date', lambda x: (snapshot - x.min()).days),
    
    # Frequency features
    total_orders          = ('order_date', 'count'),
    order_days_span       = ('order_date', lambda x: (x.max() - x.min()).days),
    
    # Monetary features
    total_revenue         = ('revenue', 'sum'),
    avg_order_value       = ('revenue', 'mean'),
    max_order_value       = ('revenue', 'max'),
    revenue_std           = ('revenue', 'std'),
    
    # Diversity features
    unique_categories     = ('category', 'nunique'),
    fav_category          = ('category', lambda x: x.mode()[0])
).round(2)

# Derived features
features['avg_days_between_orders'] = (features['order_days_span'] / 
                                        features['total_orders'].clip(lower=1)).round(1)
features['is_high_value'] = (features['total_revenue'] > features['total_revenue'].quantile(0.8)).astype(int)
features['revenue_consistency'] = (1 - features['revenue_std'] / 
                                   features['avg_order_value'].clip(lower=0.01)).round(2)

# Simulate churn label (in real life, this would be from business rules)
features['churned'] = ((features['days_since_last_order'] > 180) & 
                        (features['total_orders'] < 5)).astype(int)

print(f'Churn feature table: {features.shape}')
print(f'Churned customers  : {features["churned"].sum()} ({features["churned"].mean()*100:.1f}%)')
print('\nSample feature table:')
features.head(5).to_string()

## PRACTICAL EXERCISES

### Try these on your own — solutions follow each block.

---

In [ ]:
# ── EXERCISE DATASET ──────────────────────────────────────────────────────────
np.random.seed(100)
exercise_df = pd.DataFrame({
    'employee_id': range(1, 51),
    'name'       : [f'Employee_{i}' for i in range(1, 51)],
    'department' : np.random.choice(['Sales','Eng','Marketing','HR','Finance'], 50),
    'salary'     : np.random.randint(45000, 150000, 50),
    'years_exp'  : np.random.randint(1, 20, 50),
    'performance': np.random.choice(['Excellent','Good','Average','Poor'], 50,
                                     p=[0.15, 0.40, 0.35, 0.10]),
    'hire_date'  : pd.date_range('2015-01-01', periods=50, freq='ME')
})

print('Exercise dataset:')
exercise_df.head(5)

In [ ]:
# EXERCISE 1: Find average salary by department, sorted descending
# Your code here:

# ── SOLUTION ──
ex1 = exercise_df.groupby('department')['salary'].mean().sort_values(ascending=False).round(0)
print('EX1: Avg salary by dept:\n', ex1)

In [ ]:
# EXERCISE 2: Add a column showing each employee's salary percentile within their dept

# ── SOLUTION ──
exercise_df['salary_pctile_in_dept'] = exercise_df.groupby('department')['salary'].rank(pct=True).round(2)
print('EX2: Salary percentile within dept:')
print(exercise_df[['name','department','salary','salary_pctile_in_dept']].head(10).to_string(index=False))

In [ ]:
# EXERCISE 3: Find employees hired in Q1 of any year

# ── SOLUTION ──
q1_hires = exercise_df[exercise_df['hire_date'].dt.quarter == 1]
print(f'EX3: Q1 hires: {len(q1_hires)} employees')
print(q1_hires[['name','department','hire_date']].to_string(index=False))

In [ ]:
# EXERCISE 4: Create a performance bonus column:
#   Excellent → 15% of salary
#   Good      → 10%
#   Average   → 5%
#   Poor      → 0%

# ── SOLUTION ──
bonus_map = {'Excellent': 0.15, 'Good': 0.10, 'Average': 0.05, 'Poor': 0.0}
exercise_df['bonus'] = (exercise_df['salary'] * 
                        exercise_df['performance'].map(bonus_map)).round(0)
print('EX4: Bonus calculation:')
print(exercise_df[['name','salary','performance','bonus']].head(10).to_string(index=False))

In [ ]:
# EXERCISE 5: Which department has the highest % of Excellent performers?

# ── SOLUTION ──
ex5 = exercise_df.groupby('department').agg(
    total       = ('employee_id', 'count'),
    excellent   = ('performance', lambda x: (x == 'Excellent').sum())
)
ex5['excellent_pct'] = (ex5['excellent'] / ex5['total'] * 100).round(1)
print('EX5: Excellent % by department:')
print(ex5.sort_values('excellent_pct', ascending=False).to_string())

In [ ]:
# EXERCISE 6: Reshape the salary data into a wide format
# Rows = department, Columns = performance level, Values = avg salary

# ── SOLUTION ──
ex6 = exercise_df.pivot_table(
    values='salary',
    index='department',
    columns='performance',
    aggfunc='mean',
    fill_value=0
).round(0)
print('EX6: Avg salary by dept × performance:')
print(ex6.to_string())

---

## QUICK REFERENCE CHEAT SHEET

```python
# ── Reading Data ──────────────────────────────────────────────────
df = pd.read_csv('file.csv', parse_dates=['date'], dtype={'id':'int32'})
df = pd.read_excel('file.xlsx', sheet_name='Sheet1')
df = pd.read_parquet('file.parquet')
df = pd.read_sql("SELECT * FROM table", conn)

# ── Inspection ────────────────────────────────────────────────────
df.head() | df.tail() | df.sample(5)
df.info() | df.describe() | df.dtypes
df.isnull().sum() | df.nunique()

# ── Selection ─────────────────────────────────────────────────────
df['col']                          # Series
df[['col1','col2']]                # DataFrame
df.loc[mask, cols]                 # label-based
df.iloc[0:5, 0:3]                  # position-based
df.query("col > 100 and region=='North'")

# ── Cleaning ──────────────────────────────────────────────────────
df.dropna(how='all') | df.fillna(0) | df.fillna(method='ffill')
df.drop_duplicates(subset=['key_col'], keep='first')
df['col'] = pd.to_numeric(df['col'], errors='coerce')
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df.clip(lower=lb, upper=ub)  # cap outliers

# ── GroupBy ───────────────────────────────────────────────────────
df.groupby('col')['val'].agg(['sum','mean','count'])
df.groupby('col').agg(name=('val','sum'), cnt=('id','count'))
df.groupby('col')['val'].transform('mean')  # keep row index
df.groupby('col').filter(lambda x: x['val'].sum() > 1000)

# ── Reshape ───────────────────────────────────────────────────────
df.pivot_table(values, index, columns, aggfunc='sum', margins=True)
df.melt(id_vars=['id'], var_name='month', value_name='revenue')
df.pivot(index='row', columns='col', values='val')
df.explode('list_column')

# ── Merge ─────────────────────────────────────────────────────────
pd.merge(left, right, on='key', how='inner|left|right|outer')
pd.merge(left, right, left_on='a', right_on='b')  # different key names
pd.concat([df1, df2], ignore_index=True)           # vertical stack

# ── Time Series ───────────────────────────────────────────────────
df.set_index('date').resample('ME').sum()          # monthly
df['col'].rolling(7).mean()                        # 7-day MA
df['col'].shift(1)                                 # lag
df['col'].pct_change()                             # % change

# ── Performance ───────────────────────────────────────────────────
df['col'] = df['col'].astype('category')           # memory saving
np.where(condition, val_true, val_false)           # vectorized if-else
pd.to_numeric(df['col'], downcast='float')         # downcast dtype
df.memory_usage(deep=True).sum() / 1024**2         # memory in MB
```